# AI-Generated Text Detection with BERT
### Cleaned, Corrected & Improved version

This notebook fine-tunes **bert-base-uncased** to classify essays as human-written (`0`) or AI-generated (`1`) for the Kaggle competition *LLM Detect AI Generated Text*.

**What was fixed / improved compared to the original:**
1. ✅ Fixed the validation bug where `val_labels` was overwritten (accuracy was computed against itself).
2. ✅ Removed aggressive preprocessing (stopword removal, punctuation stripping, lowercasing) — BERT works best on **natural text**, and lowercasing is already handled by the tokenizer.
3. ✅ Removed the `nltk` dependency (a common source of runtime errors).
4. ✅ Increased `max_length` from 128 → **512** so full essays are used, with batch inference on CPU/GPU.
5. ✅ Moved `AdamW` to the stable `torch.optim` implementation, added a **learning-rate scheduler** and **gradient clipping**.
6. ✅ Added proper validation metrics: **ROC-AUC** (the competition metric), accuracy, and per-epoch model checkpointing (**best model is kept**).
7. ✅ Added **reproducibility seeds** and paths that auto-detect Kaggle vs. local machines.

## 1. Setup & Imports

Everything below is self-contained. The only external assets needed are `train_essays.csv` and `test_essays.csv` — place them in `./data/` locally, or run on Kaggle unchanged.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- Data / output locations ----------
if os.path.exists("/kaggle/input/llm-detect-ai-generated-text/train_essays.csv"):
    DATA_DIR = "/kaggle/input/llm-detect-ai-generated-text"   # Kaggle
    OUT_DIR  = "/kaggle/working"
else:
    DATA_DIR = "./data"                                        # local machine
    OUT_DIR  = "."
    os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory : {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")

# ---------- Hyper-parameters ----------
MAX_LEN    = 512      # BERT can see the full essay (was 128 before)
BATCH_SIZE = 8        # smaller batch -> fits even on modest GPUs with 512 tokens
EPOCHS     = 3        # few epochs -> avoids overfitting on ~1.4k essays
LR         = 2e-5     # standard fine-tuning learning rate for BERT

## 2. Load & Explore the Data

In [ ]:
train_essays = pd.read_csv(f"{DATA_DIR}/train_essays.csv")
test_essays  = pd.read_csv(f"{DATA_DIR}/test_essays.csv")

print(f"Train shape: {train_essays.shape} | Test shape: {test_essays.shape}")
train_essays.head()

In [ ]:
# Class distribution
ax = sns.countplot(data=train_essays, x="generated")
ax.set_xticklabels(["human (0)", "AI (1)"])
ax.set_title("Class distribution in training data")
plt.show()
print(train_essays["generated"].value_counts())

## 3. Preprocessing — keep it minimal!

**Why not remove stopwords/punctuation?** BERT was pre-trained on natural text. Stripping punctuation and stopwords destroys sentence structure that BERT uses (e.g., negations, discourse markers), and it usually *lowers* accuracy. We only normalise whitespace, and the tokenizer's `do_lower_case` handles casing.


In [ ]:
def clean_text(text: str) -> str:
    """Minimal cleaning: collapse whitespace only. BERT handles the rest."""
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

train_essays["text"] = train_essays["text"].apply(clean_text)
test_essays["text"]  = test_essays["text"].apply(clean_text)

# Stratified split keeps the class ratio in both sets
X_train, X_val, y_train, y_val = train_test_split(
    train_essays["text"], train_essays["generated"],
    test_size=0.2, random_state=SEED, stratify=train_essays["generated"]
)
print(f"Train: {len(X_train)} essays | Validation: {len(X_val)} essays")

## 4. Tokenization

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def encode(texts):
    return tokenizer(
        list(texts),
        padding="max_length",      # fixed-length tensors -> simple & safe
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

enc_train = encode(X_train)
enc_val   = encode(X_val)

train_dataset = TensorDataset(enc_train["input_ids"], enc_train["attention_mask"], torch.tensor(y_train.values))
val_dataset   = TensorDataset(enc_val["input_ids"],   enc_val["attention_mask"],   torch.tensor(y_val.values))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
print("Tokenization complete.")

## 5. Model, Optimizer & Scheduler

In [ ]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),   # warm up for 10% of steps
    num_training_steps=total_steps,
)

## 6. Training with per-epoch validation

Each epoch we validate and compute **accuracy** and **ROC-AUC**, and keep only the **best** model. No more overwriting labels — `all_labels` and `all_probs` are separate lists.


In [ ]:
best_auc = 0.0
history = []

for epoch in range(EPOCHS):
    # ---------- Training ----------
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for input_ids, attention_mask, labels in train_loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # avoid exploding gradients
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    # ---------- Validation ----------
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs  = torch.softmax(logits, dim=1)[:, 1]          # P(AI-generated)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_auc = roc_auc_score(all_labels, all_probs)
    val_acc = accuracy_score(all_labels, (np.array(all_probs) >= 0.5).astype(int))

    history.append({"epoch": epoch + 1, "train_loss": train_loss / len(train_loader),
                    "train_acc": correct / total, "val_acc": val_acc, "val_auc": val_auc})
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={history[-1]['train_loss']:.4f} "
          f"| train_acc={history[-1]['train_acc']:.4f} | val_acc={val_acc:.4f} | val_auc={val_auc:.4f}")

    # ---------- Save the best model ----------
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")
        print(f"  -> New best model saved (val_auc={best_auc:.4f})")

## 7. Final Evaluation on the Validation Set

In [ ]:
# Reload the best checkpoint before final evaluation
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location=device))
model.eval()

val_probs, val_true = [], []
with torch.no_grad():
    for input_ids, attention_mask, labels in val_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        val_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        val_true.extend(labels.numpy())

val_pred = (np.array(val_probs) >= 0.5).astype(int)
print(f"Validation ROC-AUC: {roc_auc_score(val_true, val_probs):.4f}")
print(f"Validation Accuracy: {accuracy_score(val_true, val_pred):.4f}")
print("\nClassification report:")
print(classification_report(val_true, val_pred, target_names=["human", "AI-generated"]))

## 8. Predicting the Test Set & Submission

The test set is processed **in batches** — the original code fed all test essays at once, which crashes on GPUs with limited memory.


In [ ]:
enc_test = encode(test_essays["text"])
test_dataset = TensorDataset(enc_test["input_ids"], enc_test["attention_mask"])
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

model.eval()
test_probs = []
with torch.no_grad():
    for input_ids, attention_mask in test_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        test_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())

submission = pd.DataFrame({"id": test_essays["id"], "generated": test_probs})
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print(f"Saved {len(submission)} predictions to {OUT_DIR}/submission.csv")
submission.head()

## 9. Summary

| Issue in original | Fix here |
|---|---|
| `val_labels` overwritten by predictions | Separate `all_labels` / `all_probs` lists |
| Stopword/punctuation removal | Minimal whitespace-only cleaning (better for BERT) |
| `nltk` stopwords → runtime error without download | Removed dependency entirely |
| `max_length=128` truncated essays | `max_length=512`, batch size 8 |
| 10 epochs → overfitting | 3 epochs + best-checkpoint on val ROC-AUC |
| `AdamW` from deprecated `transformers` | `torch.optim.AdamW` + LR scheduler |
| Test-set inference in one shot (OOM risk) | Batched inference |
| Kaggle-only paths | Auto-detect Kaggle vs. local `./data/` |

**If accuracy still seems low**, remember this competition's public test set is adversarial (perplexity-filtered), so even strong models score modest AUC — that is expected and a good talking point for your presentation!